In [ ]:
import bibtexparser
from bibtexparser import middlewares, model

# This notebook targets the bibtexparser 2.x API; the 1.x loads/dumps/bparser API must be absent.
assert not hasattr(bibtexparser, "loads"), "bibtexparser.loads exists: this is the 1.x API"
assert not hasattr(bibtexparser, "dumps"), "bibtexparser.dumps exists: this is the 1.x API"
for _removed in ("bparser", "bwriter", "customization"):
    try:
        __import__(f"bibtexparser.{_removed}")
    except ImportError:
        pass
    else:
        raise AssertionError(f"bibtexparser.{_removed} is importable: this is the 1.x API")

# Example 1: Basic BibTeX Parsing (v2 API)
print("Example 1: Basic BibTeX Parsing")

try:
    bibtex_str = """
    @article{einstein1905,
        title={Zur Elektrodynamik bewegter Körper},
        author={Einstein, Albert},
        journal={Annalen der Physik},
        volume={17},
        number={10},
        pages={891--921},
        year={1905},
        publisher={Wiley Online Library}
    }

    @book{knuth1997,
        title={The Art of Computer Programming},
        author={Knuth, Donald E.},
        volume={1},
        year={1997},
        publisher={Addison-Wesley}
    }
    """

    library = bibtexparser.parse_string(bibtex_str)

    print(f"✓ BibTeX parsed successfully")
    print(f"Number of entries: {len(library.entries)}")
    assert len(library.entries) == 2, [e.key for e in library.entries]
    assert not library.failed_blocks, library.failed_blocks

    for entry in library.entries:
        assert isinstance(entry, model.Entry), type(entry)
        print(f"\n📖 Entry: {entry.key}")
        print(f"  Type: {entry.entry_type}")
        print(f"  Title: {entry['title']}")
        print(f"  Author: {entry['author']}")
        print(f"  Year: {entry['year']}")

    einstein_entry = library.entries_dict['einstein1905']
    assert einstein_entry.entry_type == 'article', einstein_entry.entry_type
    assert len(einstein_entry.fields) == 8, [f.key for f in einstein_entry.fields]
    print(f"\n✓ Found Einstein entry with {len(einstein_entry.fields)} fields")

except Exception as e:
    print(f"Error in BibTeX parsing: {e}")
    raise AssertionError(
        f"bibtexparser example (cell 0) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 completed\n")

In [ ]:
import bibtexparser
from bibtexparser import model

# Example 2: Creating and Writing BibTeX (v2 API)
print("Example 2: Creating and Writing BibTeX")

try:
    # v1 built a BibDatabase of dicts; v2 builds a Library of model.Entry/model.Field objects.
    library = bibtexparser.Library()
    library.add([
        model.Entry('article', 'turing1950', [
            model.Field('title', '{Computing Machinery and Intelligence}'),
            model.Field('author', '{Turing, Alan M.}'),
            model.Field('journal', '{Mind}'),
            model.Field('year', '{1950}'),
        ]),
        model.Entry('inproceedings', 'dijkstra1968', [
            model.Field('title', '{Go To Statement Considered Harmful}'),
            model.Field('author', '{Dijkstra, Edsger W.}'),
            model.Field('booktitle', '{Communications of the ACM}'),
            model.Field('year', '{1968}'),
        ]),
        model.Entry('book', 'ritchie1988', [
            model.Field('title', '{The C Programming Language}'),
            model.Field('author', '{Kernighan, Brian W. and Ritchie, Dennis M.}'),
            model.Field('year', '{1988}'),
        ]),
    ])

    print(f"✓ Created library with {len(library.entries)} entries")
    assert len(library.entries) == 3, [e.key for e in library.entries]

    bibtex_str = bibtexparser.write_string(library)
    print("✓ Generated BibTeX:")
    print(bibtex_str[:200] + "..." if len(bibtex_str) > 200 else bibtex_str)

    # The round trip is the assertion: what we wrote must parse back to the same three entries.
    parsed = bibtexparser.parse_string(bibtex_str)
    assert not parsed.failed_blocks, parsed.failed_blocks
    assert {e.key for e in parsed.entries} == {'turing1950', 'dijkstra1968', 'ritchie1988'}, (
        [e.key for e in parsed.entries]
    )
    print(f"✓ Verification: parsed back {len(parsed.entries)} entries")

    turing_entry = parsed.entries_dict['turing1950']
    assert turing_entry.entry_type == 'article', turing_entry.entry_type
    assert 'Computing Machinery' in turing_entry['title'], turing_entry['title']
    print(f"✓ Turing entry verified: {turing_entry['title'][:30]}...")

except Exception as e:
    print(f"Error in BibTeX creation: {e}")
    raise AssertionError(
        f"bibtexparser example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 completed\n")

In [ ]:
import bibtexparser
from bibtexparser import middlewares, model

# Example 3: Middleware-driven parsing (the v2 replacement for BibTexParser attributes)
print("Example 3: Custom Parser Configuration")

try:
    complex_bibtex = """
    @article{unicode_test2024,
        title={Testing Unicode: Café, naïve, and résumé},
        author={Müller, François and García, José},
        journal={International Journal of Üñîçødé},
        year={2024},
        keywords={unicode, testing, special characters}
    }

    @inproceedings{case_test2024,
        TITLE={Mixed Case TITLE},
        Author={Smith, John AND Doe, Jane},
        BookTitle={Proceedings of the Test Conference},
        Year={2024}
    }
    """

    # v1 `parser.homogenise_fields = True` is now an explicit middleware.
    library = bibtexparser.parse_string(
        complex_bibtex,
        append_middleware=[middlewares.NormalizeFieldKeys()],
    )

    print(f"✓ Parsed {len(library.entries)} entries with middleware")
    assert len(library.entries) == 2, [e.key for e in library.entries]
    assert not library.failed_blocks, library.failed_blocks

    for entry in library.entries:
        print(f"\n📚 Entry: {entry.key}")
        print(f"  Fields: {[f.key for f in entry.fields]}")

    unicode_entry = library.entries_dict['unicode_test2024']
    assert 'Café' in unicode_entry['title'], unicode_entry['title']
    print(f"✓ Unicode title preserved: {unicode_entry['title']}")

    # NormalizeFieldKeys is what makes `TITLE`/`Author`/`BookTitle` addressable in lower case.
    case_entry = library.entries_dict['case_test2024']
    field_keys = {f.key for f in case_entry.fields}
    assert field_keys == {'title', 'author', 'booktitle', 'year'}, field_keys
    print(f"✓ Case normalization: {sorted(field_keys)}")

except Exception as e:
    print(f"Error in custom parsing: {e}")
    raise AssertionError(
        f"bibtexparser example (cell 2) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 completed\n")

In [ ]:
import bibtexparser
from bibtexparser import middlewares, model

# Example 4: Library manipulation (v2 API)
print("Example 4: Bibliography Database Manipulation")

try:
    sample_bibtex = """
    @article{paper1,
        title={First Paper},
        author={Author One},
        year={2020},
        journal={Journal A}
    }
    @article{paper2,
        title={Second Paper},
        author={Author Two},
        year={2021},
        journal={Journal B}
    }
    @book{book1,
        title={A Great Book},
        author={Book Author},
        year={2019},
        publisher={Publisher X}
    }
    """

    # v2 keeps the `{...}` delimiters in a field VALUE unless this middleware strips them --
    # `int(entry['year'])` on a parsed entry is `int('{2020}')` without it.
    library = bibtexparser.parse_string(
        sample_bibtex,
        append_middleware=[middlewares.RemoveEnclosingMiddleware()],
    )
    assert library.entries_dict['paper1']['year'] == '2020', library.entries_dict['paper1']['year']
    print(f"✓ Loaded library with {len(library.entries)} entries")
    assert len(library.entries) == 3, [e.key for e in library.entries]

    # v1 read `e['ENTRYTYPE']` / `e['ID']`; v2 exposes them as `entry_type` / `key`.
    articles = [e for e in library.entries if e.entry_type == 'article']
    books = [e for e in library.entries if e.entry_type == 'book']
    print(f"✓ Found {len(articles)} articles and {len(books)} books")
    assert (len(articles), len(books)) == (2, 1), (len(articles), len(books))

    recent_entries = [e for e in library.entries if int(e['year']) >= 2020]
    assert len(recent_entries) == 2, [e.key for e in recent_entries]
    print(f"✓ Found {len(recent_entries)} entries from 2020 or later")

    library.add(model.Entry('inproceedings', 'conference2024', [
        model.Field('title', 'New Conference Paper'),
        model.Field('author', 'Conference Author'),
        model.Field('year', '2024'),
    ]))
    assert len(library.entries) == 4, [e.key for e in library.entries]
    print(f"✓ Added new entry. Library now has {len(library.entries)} entries")

    paper1 = library.entries_dict['paper1']
    paper1.set_field(model.Field('note', 'This entry was modified'))
    assert paper1['note'] == 'This entry was modified', paper1['note']
    print(f"✓ Modified entry: {paper1.key}")

    library.remove(library.entries_dict['paper2'])
    assert 'paper2' not in library.entries_dict, sorted(library.entries_dict)
    print(f"✓ Removed entry. Library now has {len(library.entries)} entries")

    entry_types: dict[str, int] = {}
    years = []
    for entry in library.entries:
        entry_types[entry.entry_type] = entry_types.get(entry.entry_type, 0) + 1
        years.append(int(entry['year']))

    print(f"✓ Entry types: {entry_types}")
    assert entry_types == {'article': 1, 'book': 1, 'inproceedings': 1}, entry_types
    print(f"✓ Year range: {min(years)} - {max(years)}")
    assert (min(years), max(years)) == (2019, 2024), (min(years), max(years))

except Exception as e:
    print(f"Error in database manipulation: {e}")
    raise AssertionError(
        f"bibtexparser example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 completed\n")

In [ ]:
import bibtexparser
from bibtexparser import middlewares, model

# Example 5: Failed blocks, BibtexFormat writing, and @string resolution
print("Example 5: Advanced Features and Error Handling")

try:
    problematic_bibtex = """
    @article{good_entry,
        title={This is a good entry},
        author={Good Author},
        year={2024}
    }

    @article{missing_comma
        title={This entry has syntax issues}
        author={Problem Author}
        year={2024}
    }
    """

    # v2 does not throw on malformed input: it parks the block in `failed_blocks`, which is a
    # STRONGER assertion than v1's silent `ignore_nonstandard_types`, so assert on it.
    library = bibtexparser.parse_string(problematic_bibtex)
    print(f"✓ Parsed {len(library.entries)} entries, {len(library.failed_blocks)} failed block(s)")
    assert len(library.entries) == 1, [e.key for e in library.entries]
    assert len(library.failed_blocks) == 1, library.failed_blocks
    assert isinstance(library.failed_blocks[0], model.ParsingFailedBlock), type(library.failed_blocks[0])

    # v1 `BibTexWriter` attributes are now `BibtexFormat` + sorting middleware.
    bibtex_format = bibtexparser.BibtexFormat()
    bibtex_format.indent = '    '
    bibtex_format.value_column = 15
    bibtex_format.trailing_comma = True

    formatted_bibtex = bibtexparser.write_string(
        library,
        prepend_middleware=[middlewares.SortBlocksByTypeAndKeyMiddleware()],
        bibtex_format=bibtex_format,
    )
    assert '@article{good_entry' in formatted_bibtex, formatted_bibtex
    # Assert the FORMATTED entry, not a bare `'    title'` count: the failed block is written
    # back verbatim and its own source line also starts with four spaces and `title`.
    assert '\n    title        = {This is a good entry},\n' in formatted_bibtex, formatted_bibtex
    print("✓ Generated formatted BibTeX with BibtexFormat")
    for line in [ln for ln in formatted_bibtex.split('\n') if ln.strip()][:6]:
        print(f"  {line}")

    abbrev_bibtex = """
    @string{JCP = "Journal of Computational Physics"}

    @article{abbrev_test,
        title={Testing Abbreviations},
        author={Test Author},
        journal=JCP,
        year={2024}
    }
    """

    # v1 `parser.interpolate_strings` is now ResolveStringReferencesMiddleware.
    resolved = bibtexparser.parse_string(
        abbrev_bibtex,
        append_middleware=[middlewares.ResolveStringReferencesMiddleware()],
    )
    assert len(resolved.strings) == 1, resolved.strings
    journal = resolved.entries_dict['abbrev_test']['journal']
    assert journal == "Journal of Computational Physics", journal
    print(f"✓ String interpolation: {journal}")

    print("✓ Advanced features and error handling completed")

except Exception as e:
    print(f"Handled error gracefully: {e}")
    raise AssertionError(
        f"bibtexparser example (cell 4) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 completed")